In [4]:
import sys
sys.path.append('C:/Users/sliso/LastTradingProject_7052026/indicators')
sys.path.append('C:/Users/sliso/LastTradingProject_7052026/indicators/trend_strength')

In [9]:
import pandas as pd
import numpy as np
from numba import njit
import plotly.graph_objects as go
import talib as ta
from plotly.subplots import make_subplots
from SignalDecorator import signal  # your decorator
from EMMode import _ehlers_core_numba, _normalise_bandwidth


In [10]:
data=pd.read_csv("../all_data_EUR_USD.csv", parse_dates=['Date'], index_col='Date')

In [17]:
"""
calibrate_ehlers_threshold.py
─────────────────────────────
Find the optimal EhlersMarketMode threshold for a given OHLC dataset.

Usage
-----
    from calibrate_ehlers_threshold import calibrate
    results_df, best = calibrate(data)          # data = pd.DataFrame OHLCV
    print(f"Best threshold: {best:.3f}")
    results_df.to_csv("calibration_results.csv")

Three objective metrics are combined into a composite score:

  1. ADX Separation   – mean ADX in trending regime minus mean ADX in cycling
                        regime.  A good threshold creates clearly different ADX
                        distributions.  Weight 0.40.

  2. Regime Persistence – average number of consecutive bars in the same regime.
                          Too short = noisy / whipsaw.  Weight 0.35.

  3. Regime Balance    – penalises thresholds where one regime dominates.  Ideal
                          split is 40–60 % trending.  Weight 0.25.

Return-autocorrelation separation is computed as a diagnostic but not included
in the composite score (it is noisy on 15-min FX data).
"""




# ─────────────────────────────────────────────────────────────────────────────
# Internal helpers
# ─────────────────────────────────────────────────────────────────────────────

def _avg_regime_duration(mode_series: pd.Series) -> float:
    """
    Mean number of consecutive bars in the same regime state.
    Computed on the valid (non-NaN) portion of the series.
    """
    valid = mode_series.dropna().values
    if len(valid) == 0:
        return 0.0

    durations = []
    current = 1
    for i in range(1, len(valid)):
        if valid[i] == valid[i - 1]:
            current += 1
        else:
            durations.append(current)
            current = 1
    durations.append(current)
    return float(np.mean(durations))


def _minmax_norm(s: pd.Series) -> pd.Series:
    lo, hi = s.min(), s.max()
    if hi == lo:
        return pd.Series(0.5, index=s.index)
    return (s - lo) / (hi - lo)


# ─────────────────────────────────────────────────────────────────────────────
# Main calibration function
# ─────────────────────────────────────────────────────────────────────────────

def calibrate(
    data: pd.DataFrame,
    normalisation_lookback: int = 100,   # ~5 trading days on 15-min bars
    threshold_range: tuple = (0.10, 0.90),
    n_steps: int = 33,                   # step size ≈ 0.025
    adx_period: int = 14,
    weights: dict = None,
) -> tuple[pd.DataFrame, float]:
    """
    Scan threshold values and score each on three regime-quality metrics.

    Parameters
    ----------
    data : pd.DataFrame
        OHLCV with columns Open, High, Low, Close (Volume optional).
    normalisation_lookback : int
        Rolling window for bandwidth min-max normalisation.
        500 bars ≈ 5 trading days on 15-min data; tune if you change timeframe.
    threshold_range : tuple
        (min_threshold, max_threshold) to scan.
    n_steps : int
        Number of evenly-spaced threshold candidates to evaluate.
    adx_period : int
        TA-Lib ADX period.
    weights : dict, optional
        Override default weights for composite score.
        Keys: 'adx', 'persistence', 'balance'.

    Returns
    -------
    results_df : pd.DataFrame
        One row per threshold with all metrics and composite score.
    best_threshold : float
        Threshold with the highest composite score.
    """
    if weights is None:
        weights = {'adx': 0.40, 'persistence': 0.35, 'balance': 0.25}

    # ── Step 1: compute normalised bandwidth once ─────────────────────────────
    close_arr = data['Close'].values.astype(np.float64)
    raw_bw    = pd.Series(
        _ehlers_core_numba(close_arr), index=data.index, dtype=float
    )
    norm_bw = _normalise_bandwidth(raw_bw, normalisation_lookback)

    # ── Step 2: pre-compute ADX once ─────────────────────────────────────────
    adx = pd.Series(
        ta.ADX(
            data['High'].values.astype(np.float64),
            data['Low'].values.astype(np.float64),
            close_arr,
            timeperiod=adx_period
        ),
        index=data.index
    )

    # ── Step 3: pre-compute 1-bar return autocorrelation helper ──────────────
    ret = data['Close'].pct_change()

    valid_mask = norm_bw.notna() & adx.notna() & ret.notna()

    # ── Step 4: threshold scan ────────────────────────────────────────────────
    thresholds = np.linspace(threshold_range[0], threshold_range[1], n_steps)
    rows = []

    for thr in thresholds:
        # Classify: bandwidth_norm > thr → strong cycle → CYCLING (0)
        #           bandwidth_norm ≤ thr → weak   cycle → TRENDING (1)
        mode = pd.Series(np.nan, index=data.index)
        mode[valid_mask] = np.where(norm_bw[valid_mask] > thr, 0.0, 1.0)

        trending = mode == 1.0
        cycling  = mode == 0.0

        n_trend = trending.sum()
        n_cycle = cycling.sum()
        n_total = n_trend + n_cycle

        if n_total == 0 or n_trend == 0 or n_cycle == 0:
            continue

        pct_trending = n_trend / n_total

        # Metric 1 – ADX separation
        adx_trend = adx[trending].mean()
        adx_cycle = adx[cycling].mean()
        adx_sep   = adx_trend - adx_cycle   # higher = regime more distinct

        # Metric 2 – regime persistence (avg consecutive bars in same state)
        persistence = _avg_regime_duration(mode)

        # Metric 3 – balance penalty (want pct_trending near 0.40–0.60)
        balance_score = max(0.0, 1.0 - 3.0 * abs(pct_trending - 0.50))

        # Diagnostic – return autocorrelation separation (not in composite)
        ac_trend = ret[trending].autocorr(lag=1)
        ac_cycle = ret[cycling].autocorr(lag=1)

        rows.append(dict(
            threshold       = round(float(thr), 4),
            pct_trending    = round(pct_trending, 4),
            adx_trending    = round(adx_trend, 4),
            adx_cycling     = round(adx_cycle, 4),
            adx_separation  = round(adx_sep, 4),
            persistence_bars= round(persistence, 2),
            balance_score   = round(balance_score, 4),
            autocorr_trend  = round(ac_trend, 4),
            autocorr_cycle  = round(ac_cycle, 4),
        ))

    results = pd.DataFrame(rows)

    # ── Step 5: normalise each metric to [0, 1] and combine ──────────────────
    results['norm_adx']         = _minmax_norm(results['adx_separation'])
    results['norm_persistence'] = _minmax_norm(results['persistence_bars'])
    results['norm_balance']     = _minmax_norm(results['balance_score'])

    results['composite_score'] = (
        weights['adx']         * results['norm_adx']         +
        weights['persistence'] * results['norm_persistence'] +
        weights['balance']     * results['norm_balance']
    )

    best_threshold = float(results.loc[results['composite_score'].idxmax(), 'threshold'])

    return results, best_threshold


# ─────────────────────────────────────────────────────────────────────────────
# Visualisation
# ─────────────────────────────────────────────────────────────────────────────

def plot_calibration(results: pd.DataFrame, best_threshold: float) -> None:
    """
    Four-panel Plotly chart showing all metrics and the composite score
    across the threshold grid.  Vertical red line marks the best threshold.
    """
    fig = make_subplots(
        rows=4, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.06,
        subplot_titles=(
            'ADX Separation  (trending ADX − cycling ADX)',
            'Regime Persistence  (avg consecutive bars)',
            '% Time Trending  (target band 40–60 %)',
            'Composite Score  ← optimal threshold here',
        )
    )

    kw = dict(mode='lines+markers', marker=dict(size=5))

    fig.add_trace(go.Scatter(
        x=results['threshold'], y=results['adx_separation'],
        line=dict(color='dodgerblue', width=2), name='ADX sep.', **kw
    ), row=1, col=1)

    fig.add_trace(go.Scatter(
        x=results['threshold'], y=results['persistence_bars'],
        line=dict(color='darkorange', width=2), name='Persistence', **kw
    ), row=2, col=1)

    fig.add_trace(go.Scatter(
        x=results['threshold'], y=results['pct_trending'] * 100,
        line=dict(color='mediumpurple', width=2), name='% Trending', **kw
    ), row=3, col=1)

    # Target band shading 40–60 %
    fig.add_hrect(y0=40, y1=60, fillcolor='rgba(0,255,0,0.08)',
                  opacity=1.0, line_width=0, row=3, col=1)

    fig.add_trace(go.Scatter(
        x=results['threshold'], y=results['composite_score'],
        line=dict(color='gold', width=3), name='Composite', **kw
    ), row=4, col=1)

    # Best threshold vertical line on all panels
    for row in range(1, 5):
        fig.add_vline(
            x=best_threshold, line_dash='dash', line_color='red',
            annotation_text=f'Best: {best_threshold:.3f}' if row == 4 else '',
            annotation_position='top right',
            row=row, col=1
        )

    fig.update_layout(
        title=f'EhlersMarketMode Threshold Calibration — best = {best_threshold:.3f}',
        height=950, width=1100,
        template='plotly_dark', hovermode='x unified',
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
    )
    fig.update_xaxes(title_text='Threshold', row=4, col=1)
    fig.update_yaxes(title_text='ADX pts',    row=1, col=1)
    fig.update_yaxes(title_text='Bars',       row=2, col=1)
    fig.update_yaxes(title_text='%',          row=3, col=1)
    fig.update_yaxes(title_text='Score',      row=4, col=1)
    fig.show()


# ─────────────────────────────────────────────────────────────────────────────
# Convenience wrapper
# ─────────────────────────────────────────────────────────────────────────────

def calibrate_and_plot(data: pd.DataFrame, **kwargs) -> tuple[pd.DataFrame, float]:
    """
    Run calibration and immediately show the diagnostic chart.

    Example
    -------
    results, best = calibrate_and_plot(eurusd_15min)
    print(results[['threshold','adx_separation','persistence_bars',
                   'pct_trending','composite_score']].to_string())
    """
    results, best = calibrate(data, **kwargs)
    plot_calibration(results, best)

    best_row = results.loc[results['threshold'] == best].iloc[0]
    print("\n── Calibration summary ────────────────────────────────────────")
    print(f"  Best threshold        : {best:.4f}")
    print(f"  ADX separation        : {best_row['adx_separation']:.2f} pts")
    print(f"  Regime persistence    : {best_row['persistence_bars']:.1f} bars")
    print(f"  % time trending       : {best_row['pct_trending']*100:.1f} %")
    print(f"  Composite score       : {best_row['composite_score']:.4f}")
    print("───────────────────────────────────────────────────────────────\n")

    return results, best


In [18]:
results, best = calibrate_and_plot(data)


── Calibration summary ────────────────────────────────────────
  Best threshold        : 0.1500
  ADX separation        : -2.48 pts
  Regime persistence    : 6.0 bars
  % time trending       : 47.9 %
  Composite score       : 0.6286
───────────────────────────────────────────────────────────────

